<a href="https://colab.research.google.com/github/vkmnamit/hack_14nov/blob/main/hack_14_nov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Prepare Colab & Inspect Dataset
We begin by verifying the GPU availability and inspecting the structure of the dataset arrays stored in the `.npz` file.

In [1]:
import numpy as np
import torch

# Check PyTorch GPU availability
gpu_available = torch.cuda.is_available()
print(f"PyTorch GPU available: {gpu_available}")
if gpu_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

# Load the dataset
path = '/content/majuli_anuga_convlstm_v2.npz'
try:
    data = np.load(path, allow_pickle=False)
    print("\nSuccessfully loaded dataset!")
    print("Keys in dataset:", list(data.keys()))

    # Display shapes of available arrays
    for key in data.keys():
        print(f"- Array '{key}': shape = {data[key].shape}, dtype = {data[key].dtype}")

    # Count scenario splits if split exists
    if 'split' in data:
        splits, counts = np.unique(data['split'], return_counts=True)
        print("\nSplit distribution (Scenarios):\n", dict(zip(splits, counts)))
except Exception as e:
    print(f"Error loading dataset: {e}")

PyTorch GPU available: False

Successfully loaded dataset!
Keys in dataset: ['state_sequence', 'initial_state', 'raw_rainfall_mm_hr', 'rainfall_effective_m_s', 'infiltration_loss_mm_hr', 'elevation_m', 'manning_n', 'wet_mask_1cm', 'times_seconds', 'scenario_ids', 'scenario_names', 'temporal_profiles', 'spatial_patterns', 'initial_condition_names', 'rainfall_profiles_mm_hr', 'split', 'mass_balance_relative_error', 'state_channel_names', 'raw_rainfall_units', 'effective_rainfall_units', 'input_channel_names', 'normalization_state_mean', 'normalization_state_std', 'normalization_raw_rainfall_mean_mm_hr', 'normalization_raw_rainfall_std_mm_hr', 'normalization_effective_rainfall_mean_m_s', 'normalization_effective_rainfall_std_m_s', 'normalization_infiltration_mean_mm_hr', 'normalization_infiltration_std_mm_hr', 'normalization_manning_mean', 'normalization_manning_std', 'normalization_elevation_mean_m', 'normalization_elevation_std_m']
- Array 'state_sequence': shape = (72, 61, 3, 50, 50), 

```markdown
## 2. Build the Model Inputs and Targets
In this step, we will:
1. Extract and normalize our variables using the pre-saved global/training-only statistics in the archive.
2. Construct the 7-channel input layout: `[h, hu, hv, raw_rainfall, infiltration_loss, elevation, manning_n]`.
3. Keep the original scenario splits intact (36 train, 18 validation, 18 test) to prevent temporal leakage.
4. Create one-step temporal window pairs (frames `0-59` as input, frames `1-60` as target).
5. Package everything into clean PyTorch `Dataset` and `DataLoader` classes.
```

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader

# Extract normalization parameters
state_mean = data['normalization_state_mean']  # shape (3,)
state_std = data['normalization_state_std']    # shape (3,)

rain_mean = data['normalization_raw_rainfall_mean_mm_hr']
rain_std = data['normalization_raw_rainfall_std_mm_hr']

infil_mean = data['normalization_infiltration_mean_mm_hr']
infil_std = data['normalization_infiltration_std_mm_hr']

elev_mean = data['normalization_elevation_mean_m']
elev_std = data['normalization_elevation_std_m']

mann_mean = data['normalization_manning_mean']
mann_std = data['normalization_manning_std']

print("Normalization values extracted.")

Normalization values extracted.


In [3]:
class MajuliDataset(Dataset):
    def __init__(self, data, split_name='train'):
        self.split_name = split_name

        # Identify indices belonging to this split
        splits = data['split']
        self.indices = np.where(splits == split_name)[0]

        # Retrieve arrays
        self.state_seq = data['state_sequence'][self.indices]  # (N, 61, 3, 50, 50)
        self.raw_rain = data['raw_rainfall_mm_hr'][self.indices]  # (N, 61, 50, 50)
        self.infil = data['infiltration_loss_mm_hr'][self.indices]  # (N, 50, 50)
        self.elev = data['elevation_m']  # (50, 50)
        self.manning = data['manning_n'][self.indices]  # (N, 50, 50)
        self.wet_mask = data['wet_mask_1cm'][self.indices]  # (N, 61, 50, 50)

        # Dimensions
        self.num_scenarios = len(self.indices)
        self.num_frames = self.state_seq.shape[1]  # 61

    def __len__(self):
        # For each scenario, we have 60 transition pairs (0 -> 1, ..., 59 -> 60)
        return self.num_scenarios * (self.num_frames - 1)

    def __getitem__(self, idx):
        # Map flat 1D index to (scenario_idx, frame_idx)
        scenario_idx = idx // (self.num_frames - 1)
        frame_idx = idx % (self.num_frames - 1)

        # 1. State current [h, hu, hv]
        state_curr = self.state_seq[scenario_idx, frame_idx].copy()
        # Normalize state channel-wise
        for c in range(3):
            state_curr[c] = (state_curr[c] - state_mean[c]) / (state_std[c] + 1e-8)

        # 2. Forcing/Static parameters
        rain = self.raw_rain[scenario_idx, frame_idx]
        rain_norm = (rain - rain_mean) / (rain_std + 1e-8)

        infil = self.infil[scenario_idx]
        infil_norm = (infil - infil_mean) / (infil_std + 1e-8)

        elev = self.elev
        elev_norm = (elev - elev_mean) / (elev_std + 1e-8)

        mann = self.manning[scenario_idx]
        mann_norm = (mann - mann_mean) / (mann_std + 1e-8)

        # Stack everything into 7 input channels
        # Shape will be (7, 50, 50)
        input_tensor = np.zeros((7, 50, 50), dtype=np.float32)
        input_tensor[0:3] = state_curr
        input_tensor[3] = rain_norm
        input_tensor[4] = infil_norm
        input_tensor[5] = elev_norm
        input_tensor[6] = mann_norm

        # Target state at frame_idx + 1 [h, hu, hv] (normalized)
        state_next = self.state_seq[scenario_idx, frame_idx + 1].copy()
        state_next_norm = np.zeros_like(state_next)
        for c in range(3):
            state_next_norm[c] = (state_next[c] - state_mean[c]) / (state_std[c] + 1e-8)

        # Mask for loss calculation
        mask = self.wet_mask[scenario_idx, frame_idx + 1]

        return (
            torch.tensor(input_tensor, dtype=torch.float32),
            torch.tensor(state_next_norm, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.bool)
        )

# Initialize Datasets
train_dataset = MajuliDataset(data, 'train')
val_dataset = MajuliDataset(data, 'validation')
test_dataset = MajuliDataset(data, 'test')

# Wrap with DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Dataset setup complete!")
print(f"Train pairs: {len(train_dataset)} | Val pairs: {len(val_dataset)} | Test pairs: {len(test_dataset)}")

Dataset setup complete!
Train pairs: 2160 | Val pairs: 1080 | Test pairs: 1080


# 3. Establish Baselines
We will evaluate:
1. **Persistence Baseline**: assuming state $t+1$ is equal to state $t$.
2. **Simple CNN Baseline**: standard 2D convolution network using the 7 channels to predict the next 3 channels.

In [4]:
import torch.nn as nn
import torch.optim as optim

def evaluate_persistence(loader):
    total_loss_mse = 0.0
    total_loss_mae = 0.0
    num_samples = 0

    for inputs, targets, _ in loader:
        current_state = inputs[:, 0:3, :, :]
        mse = nn.functional.mse_loss(current_state, targets, reduction='sum')
        mae = nn.functional.l1_loss(current_state, targets, reduction='sum')

        total_loss_mse += mse.item()
        total_loss_mae += mae.item()
        num_samples += targets.numel()

    mean_mse = total_loss_mse / num_samples
    mean_mae = total_loss_mae / num_samples
    return mean_mse, mean_mae

pers_mse, pers_mae = evaluate_persistence(val_loader)
print(f"[Persistence Baseline] Val MSE: {pers_mse:.6f} | Val MAE: {pers_mae:.6f}")

[Persistence Baseline] Val MSE: 0.070368 | Val MAE: 0.032099


In [5]:
class SimpleCNNBaseline(nn.Module):
    def __init__(self, in_channels=7, out_channels=3):
        super(SimpleCNNBaseline, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, out_channels, kernel_size=3, padding=1)
        )

    def forward(self, x):
        return self.net(x)

model_cnn = SimpleCNNBaseline()
test_input = torch.randn(8, 7, 50, 50)
test_out = model_cnn(test_input)
print(f"Simple CNN output shape: {test_out.shape} (Expected: [8, 3, 50, 50])")

Simple CNN output shape: torch.Size([8, 3, 50, 50]) (Expected: [8, 3, 50, 50])


# 3. Establish Baselines
We will now establish two baselines:
1. **Persistence Baseline**: Assumes no state change ($S_{t+1} = S_t$).
2. **Simple CNN Baseline**: Uses a feed-forward CNN (no recurrent memory) predicting one step ahead from the 7-channel input.

In [6]:
import torch.nn as nn
import torch.optim as optim

# 1. Persistence Baseline Evaluation
def evaluate_persistence(loader):
    total_loss_mse = 0.0
    total_loss_mae = 0.0
    num_samples = 0

    for inputs, targets, _ in loader:
        # Current state is stored in channels 0, 1, 2 of inputs
        current_state = inputs[:, 0:3, :, :]

        # Compute difference with target next state
        mse = nn.functional.mse_loss(current_state, targets, reduction='sum')
        mae = nn.functional.l1_loss(current_state, targets, reduction='sum')

        total_loss_mse += mse.item()
        total_loss_mae += mae.item()
        num_samples += targets.numel()

    mean_mse = total_loss_mse / num_samples
    mean_mae = total_loss_mae / num_samples
    return mean_mse, mean_mae

pers_mse, pers_mae = evaluate_persistence(val_loader)
print(f"[Persistence Baseline] Val MSE: {pers_mse:.6f} | Val MAE: {pers_mae:.6f}")

[Persistence Baseline] Val MSE: 0.070368 | Val MAE: 0.032099


In [7]:
class SimpleCNNBaseline(nn.Module):
    def __init__(self, in_channels=7, out_channels=3):
        super(SimpleCNNBaseline, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, out_channels, kernel_size=3, padding=1)
        )

    def forward(self, x):
        return self.net(x)

# Test a forward pass
model_cnn = SimpleCNNBaseline()
test_input = torch.randn(8, 7, 50, 50)
test_out = model_cnn(test_input)
print(f"Simple CNN output shape: {test_out.shape} (Expected: [8, 3, 50, 50])")

Simple CNN output shape: torch.Size([8, 3, 50, 50]) (Expected: [8, 3, 50, 50])


## Train the Simple CNN Baseline
We will now train the `SimpleCNNBaseline` model using MSE loss on the normalized states. This provides a learned baseline performance.

In [8]:
import time
import torch.nn as nn
import torch.optim as optim

# Choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model_cnn = SimpleCNNBaseline().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=1e-3)

# Training loop
epochs = 3
for epoch in range(epochs):
    model_cnn.train()
    train_loss = 0.0
    start_time = time.time()

    for inputs, targets, _ in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model_cnn(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation
    model_cnn.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets, _ in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model_cnn(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(val_loader.dataset)

    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{epochs} | Train MSE Loss: {train_loss:.6f} | Val MSE Loss: {val_loss:.6f} | Time: {epoch_time:.2f}s")

Training on device: cpu
Epoch 1/3 | Train MSE Loss: 0.373494 | Val MSE Loss: 0.111465 | Time: 15.92s
Epoch 2/3 | Train MSE Loss: 0.135815 | Val MSE Loss: 0.083324 | Time: 16.40s
Epoch 3/3 | Train MSE Loss: 0.120786 | Val MSE Loss: 0.076630 | Time: 18.58s


## Train the Simple CNN Baseline
We will now train the `SimpleCNNBaseline` model using MSE loss on the normalized states. This provides a learned baseline performance.

In [9]:
import time

# Choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model_cnn = SimpleCNNBaseline().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=1e-3)

# Training loop
epochs = 3
for epoch in range(epochs):
    model_cnn.train()
    train_loss = 0.0
    start_time = time.time()

    for inputs, targets, _ in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model_cnn(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation
    model_cnn.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets, _ in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model_cnn(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(val_loader.dataset)

    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{epochs} | Train MSE Loss: {train_loss:.6f} | Val MSE Loss: {val_loss:.6f} | Time: {epoch_time:.2f}s")

Training on device: cpu
Epoch 1/3 | Train MSE Loss: 0.432066 | Val MSE Loss: 0.118130 | Time: 15.59s
Epoch 2/3 | Train MSE Loss: 0.136826 | Val MSE Loss: 0.082649 | Time: 16.60s
Epoch 3/3 | Train MSE Loss: 0.121640 | Val MSE Loss: 0.075673 | Time: 16.17s


## Train the Simple CNN Baseline
We will now train the `SimpleCNNBaseline` model using MSE loss on the normalized states. This provides a learned baseline performance.

In [10]:
import time

# Choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model_cnn = SimpleCNNBaseline().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=1e-3)

# Training loop
epochs = 3
for epoch in range(epochs):
    model_cnn.train()
    train_loss = 0.0
    start_time = time.time()

    for inputs, targets, _ in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model_cnn(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation
    model_cnn.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets, _ in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model_cnn(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(val_loader.dataset)

    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{epochs} | Train MSE Loss: {train_loss:.6f} | Val MSE Loss: {val_loss:.6f} | Time: {epoch_time:.2f}s")

Training on device: cpu
Epoch 1/3 | Train MSE Loss: 0.405288 | Val MSE Loss: 0.112789 | Time: 15.63s
Epoch 2/3 | Train MSE Loss: 0.137512 | Val MSE Loss: 0.082338 | Time: 15.59s
Epoch 3/3 | Train MSE Loss: 0.120967 | Val MSE Loss: 0.078221 | Time: 15.52s


## Train the Simple CNN Baseline
We will now train the `SimpleCNNBaseline` model using MSE loss on the normalized states. This provides a learned baseline performance.

In [11]:
import time

# Choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model_cnn = SimpleCNNBaseline().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=1e-3)

# Training loop
epochs = 3
for epoch in range(epochs):
    model_cnn.train()
    train_loss = 0.0
    start_time = time.time()

    for inputs, targets, _ in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model_cnn(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation
    model_cnn.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets, _ in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model_cnn(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(val_loader.dataset)

    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{epochs} | Train MSE Loss: {train_loss:.6f} | Val MSE Loss: {val_loss:.6f} | Time: {epoch_time:.2f}s")

Training on device: cpu
Epoch 1/3 | Train MSE Loss: 0.401334 | Val MSE Loss: 0.117838 | Time: 16.21s
Epoch 2/3 | Train MSE Loss: 0.138175 | Val MSE Loss: 0.084726 | Time: 15.49s
Epoch 3/3 | Train MSE Loss: 0.122062 | Val MSE Loss: 0.076637 | Time: 15.56s


# 3. ConvLSTM Surrogate Model
We will now construct the ConvLSTM layer and model. The architecture processes the spatial input sequence step-by-step, preserving spatial structures via spatial convolutions within the LSTM gates.

In [12]:
import torch
import torch.nn as nn

class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super(ConvLSTMCell, self).__init__()
        self.padding = kernel_size // 2
        self.gates = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size=kernel_size,
            padding=self.padding
        )

    def forward(self, x, state):
        if state is None:
            batch_size, _, height, width = x.size()
            h = torch.zeros(batch_size, self.gates.out_channels // 4, height, width, device=x.device)
            c = torch.zeros(batch_size, self.gates.out_channels // 4, height, width, device=x.device)
        else:
            h, c = state

        combined = torch.cat([x, h], dim=1)
        gates = self.gates(combined)

        ingate, forgetgate, cellgate, outgate = torch.chunk(gates, 4, dim=1)

        ingate = torch.sigmoid(ingate)
        forgetgate = torch.sigmoid(forgetgate)
        cellgate = torch.tanh(cellgate)
        outgate = torch.sigmoid(outgate)

        c_next = (forgetgate * c) + (ingate * cellgate)
        h_next = outgate * torch.tanh(c_next)

        return h_next, (h_next, c_next)

class ConvLSTMSurrogate(nn.Module):
    def __init__(self, in_channels=7, hidden_channels=32, out_channels=3):
        super(ConvLSTMSurrogate, self).__init__()
        self.cell = ConvLSTMCell(in_channels, hidden_channels)
        self.output_layer = nn.Conv2d(hidden_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, x_seq):
        # Expect x_seq of shape (Batch, Time, Channel, Height, Width)
        batch_size, seq_len, _, h, w = x_seq.size()
        state = None

        for t in range(seq_len):
            h_state, state = self.cell(x_seq[:, t], state)

        # Decode final hidden state to predict the target variables [h, hu, hv]
        out = self.output_layer(h_state)
        return out

# Quick execution verification
model_lstm = ConvLSTMSurrogate()
test_seq_input = torch.randn(4, 5, 7, 50, 50)  # Batch=4, Time=5, Channel=7, H=50, W=50
test_lstm_out = model_lstm(test_seq_input)
print(f"ConvLSTM output shape: {test_lstm_out.shape} (Expected: [4, 3, 50, 50])")

ConvLSTM output shape: torch.Size([4, 3, 50, 50]) (Expected: [4, 3, 50, 50])


# 3. ConvLSTM Surrogate Model
We will now construct the ConvLSTM layer and model. The architecture processes the spatial input sequence step-by-step, preserving spatial structures via spatial convolutions within the LSTM gates.

In [13]:
import torch
import torch.nn as nn

class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super(ConvLSTMCell, self).__init__()
        self.padding = kernel_size // 2
        self.gates = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size=kernel_size,
            padding=self.padding
        )

    def forward(self, x, state):
        if state is None:
            batch_size, _, height, width = x.size()
            h = torch.zeros(batch_size, self.gates.out_channels // 4, height, width, device=x.device)
            c = torch.zeros(batch_size, self.gates.out_channels // 4, height, width, device=x.device)
        else:
            h, c = state

        combined = torch.cat([x, h], dim=1)
        gates = self.gates(combined)

        ingate, forgetgate, cellgate, outgate = torch.chunk(gates, 4, dim=1)

        ingate = torch.sigmoid(ingate)
        forgetgate = torch.sigmoid(forgetgate)
        cellgate = torch.tanh(cellgate)
        outgate = torch.sigmoid(outgate)

        c_next = (forgetgate * c) + (ingate * cellgate)
        h_next = outgate * torch.tanh(c_next)

        return h_next, (h_next, c_next)

class ConvLSTMSurrogate(nn.Module):
    def __init__(self, in_channels=7, hidden_channels=32, out_channels=3):
        super(ConvLSTMSurrogate, self).__init__()
        self.cell = ConvLSTMCell(in_channels, hidden_channels)
        self.output_layer = nn.Conv2d(hidden_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, x_seq):
        # Expect x_seq of shape (Batch, Time, Channel, Height, Width)
        batch_size, seq_len, _, h, w = x_seq.size()
        state = None

        for t in range(seq_len):
            h_state, state = self.cell(x_seq[:, t], state)

        # Decode final hidden state to predict the target variables [h, hu, hv]
        out = self.output_layer(h_state)
        return out

# Quick execution verification
model_lstm = ConvLSTMSurrogate()
test_seq_input = torch.randn(4, 5, 7, 50, 50)  # Batch=4, Time=5, Channel=7, H=50, W=50
test_lstm_out = model_lstm(test_seq_input)
print(f"ConvLSTM output shape: {test_lstm_out.shape} (Expected: [4, 3, 50, 50])")

ConvLSTM output shape: torch.Size([4, 3, 50, 50]) (Expected: [4, 3, 50, 50])


# 3. ConvLSTM Surrogate Model
We will now construct the ConvLSTM layer and model. The architecture processes the spatial input sequence step-by-step, preserving spatial structures via spatial convolutions within the LSTM gates.

In [14]:
import torch
import torch.nn as nn

class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super(ConvLSTMCell, self).__init__()
        self.padding = kernel_size // 2
        self.gates = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size=kernel_size,
            padding=self.padding
        )

    def forward(self, x, state):
        if state is None:
            batch_size, _, height, width = x.size()
            h = torch.zeros(batch_size, self.gates.out_channels // 4, height, width, device=x.device)
            c = torch.zeros(batch_size, self.gates.out_channels // 4, height, width, device=x.device)
        else:
            h, c = state

        combined = torch.cat([x, h], dim=1)
        gates = self.gates(combined)

        ingate, forgetgate, cellgate, outgate = torch.chunk(gates, 4, dim=1)

        ingate = torch.sigmoid(ingate)
        forgetgate = torch.sigmoid(forgetgate)
        cellgate = torch.tanh(cellgate)
        outgate = torch.sigmoid(outgate)

        c_next = (forgetgate * c) + (ingate * cellgate)
        h_next = outgate * torch.tanh(c_next)

        return h_next, (h_next, c_next)

class ConvLSTMSurrogate(nn.Module):
    def __init__(self, in_channels=7, hidden_channels=32, out_channels=3):
        super(ConvLSTMSurrogate, self).__init__()
        self.cell = ConvLSTMCell(in_channels, hidden_channels)
        self.output_layer = nn.Conv2d(hidden_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, x_seq):
        # Expect x_seq of shape (Batch, Time, Channel, Height, Width)
        batch_size, seq_len, _, h, w = x_seq.size()
        state = None

        for t in range(seq_len):
            h_state, state = self.cell(x_seq[:, t], state)

        # Decode final hidden state to predict the target variables [h, hu, hv]
        out = self.output_layer(h_state)
        return out

# Quick execution verification
model_lstm = ConvLSTMSurrogate()
test_seq_input = torch.randn(4, 5, 7, 50, 50)  # Batch=4, Time=5, Channel=7, H=50, W=50
test_lstm_out = model_lstm(test_seq_input)
print(f"ConvLSTM output shape: {test_lstm_out.shape} (Expected: [4, 3, 50, 50])")

ConvLSTM output shape: torch.Size([4, 3, 50, 50]) (Expected: [4, 3, 50, 50])


# 3. ConvLSTM Surrogate Model
We will now construct the ConvLSTM layer and model. The architecture processes the spatial input sequence step-by-step, preserving spatial structures via spatial convolutions within the LSTM gates.

In [15]:
import torch
import torch.nn as nn

class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super(ConvLSTMCell, self).__init__()
        self.padding = kernel_size // 2
        self.gates = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size=kernel_size,
            padding=self.padding
        )

    def forward(self, x, state):
        if state is None:
            batch_size, _, height, width = x.size()
            h = torch.zeros(batch_size, self.gates.out_channels // 4, height, width, device=x.device)
            c = torch.zeros(batch_size, self.gates.out_channels // 4, height, width, device=x.device)
        else:
            h, c = state

        combined = torch.cat([x, h], dim=1)
        gates = self.gates(combined)

        ingate, forgetgate, cellgate, outgate = torch.chunk(gates, 4, dim=1)

        ingate = torch.sigmoid(ingate)
        forgetgate = torch.sigmoid(forgetgate)
        cellgate = torch.tanh(cellgate)
        outgate = torch.sigmoid(outgate)

        c_next = (forgetgate * c) + (ingate * cellgate)
        h_next = outgate * torch.tanh(c_next)

        return h_next, (h_next, c_next)

class ConvLSTMSurrogate(nn.Module):
    def __init__(self, in_channels=7, hidden_channels=32, out_channels=3):
        super(ConvLSTMSurrogate, self).__init__()
        self.cell = ConvLSTMCell(in_channels, hidden_channels)
        self.output_layer = nn.Conv2d(hidden_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, x_seq):
        # Expect x_seq of shape (Batch, Time, Channel, Height, Width)
        batch_size, seq_len, _, h, w = x_seq.size()
        state = None

        for t in range(seq_len):
            h_state, state = self.cell(x_seq[:, t], state)

        # Decode final hidden state to predict the target variables [h, hu, hv]
        out = self.output_layer(h_state)
        return out

# Quick execution verification
model_lstm = ConvLSTMSurrogate()
test_seq_input = torch.randn(4, 5, 7, 50, 50)  # Batch=4, Time=5, Channel=7, H=50, W=50
test_lstm_out = model_lstm(test_seq_input)
print(f"ConvLSTM output shape: {test_lstm_out.shape} (Expected: [4, 3, 50, 50])")

ConvLSTM output shape: torch.Size([4, 3, 50, 50]) (Expected: [4, 3, 50, 50])


# 4. Train ConvLSTM with Physics-Informed Masking
We will now set up the training routine for our `ConvLSTMSurrogate` model. In accordance with Step 4, we use the wet mask to calculate loss on depth and momentum channels accurately, protecting velocity metrics from instabilities in dry regions.

In [16]:
import time
import torch.nn as nn
import torch.optim as optim

# Configure model and device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_lstm = ConvLSTMSurrogate(in_channels=7, hidden_channels=32, out_channels=3).to(device)
optimizer = optim.Adam(model_lstm.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Quick training simulation over 2 epochs on sequence inputs
epochs = 2
for epoch in range(epochs):
    model_lstm.train()
    epoch_loss = 0.0
    start_time = time.time()

    # Note: To pass sequential frames into ConvLSTM, we prepare a mini-sequence [Batch, Time, Channel, H, W]
    # Here we simulate with one temporal transition pair
    for inputs, targets, masks in train_loader:
        inputs = inputs.to(device)  # (Batch, 7, 50, 50)
        targets = targets.to(device)  # (Batch, 3, 50, 50)
        masks = masks.to(device)  # (Batch, 50, 50)

        # Add temporary sequence dimension for one-step LSTM processing: [Batch, Time=1, Channel, H, W]
        inputs_seq = inputs.unsqueeze(1)

        optimizer.zero_grad()
        outputs = model_lstm(inputs_seq)

        # Apply supervised MSE loss
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * inputs.size(0)

    epoch_loss /= len(train_loader.dataset)
    epoch_time = time.time() - start_time
    print(f"ConvLSTM Epoch {epoch+1}/{epochs} | Train MSE Loss: {epoch_loss:.6f} | Time: {epoch_time:.2f}s")

ConvLSTM Epoch 1/2 | Train MSE Loss: 0.656077 | Time: 34.79s
ConvLSTM Epoch 2/2 | Train MSE Loss: 0.238172 | Time: 34.29s


## 5. Evaluate Multi-step Stability (Rollouts)
We will now write an evaluation script that performs a recursive multi-step rollout forecast. For a given scenario, we start with the initial frame's state, and recursively predict the next states. We will report the depth MAE/RMSE and velocity errors only in cells where the water depth is $\ge 1$ cm.

In [17]:
def run_rollout_evaluation(model, dataset_data, scenario_idx, steps=60):
    model.eval()
    # Retrieve ground truth state sequence for this scenario
    # Shape: (61, 3, 50, 50)
    gt_sequence = dataset_data['state_sequence'][scenario_idx]
    raw_rain = dataset_data['raw_rainfall_mm_hr'][scenario_idx]  # (61, 50, 50)
    infil = dataset_data['infiltration_loss_mm_hr'][scenario_idx]  # (50, 50)
    elev = dataset_data['elevation_m']  # (50, 50)
    manning = dataset_data['manning_n'][scenario_idx]  # (50, 50)
    wet_mask_seq = dataset_data['wet_mask_1cm'][scenario_idx]  # (61, 50, 50)

    # Normalize static parameters
    infil_norm = (infil - infil_mean) / (infil_std + 1e-8)
    elev_norm = (elev - elev_mean) / (elev_std + 1e-8)
    manning_norm = (manning - mann_mean) / (mann_std + 1e-8)

    # Initialize prediction sequence with the first frame (frame 0)
    pred_states = [gt_sequence[0].copy()]

    with torch.no_grad():
        current_state = gt_sequence[0].copy()
        for t in range(steps):
            # Normalize current state estimate
            norm_state = np.zeros_like(current_state)
            for c in range(3):
                norm_state[c] = (current_state[c] - state_mean[c]) / (state_std[c] + 1e-8)

            # Normalize rainfall at current step
            rain_norm = (raw_rain[t] - rain_mean) / (rain_std + 1e-8)

            # Form 7-channel input
            input_tensor = np.zeros((1, 1, 7, 50, 50), dtype=np.float32)  # (Batch=1, Time=1, Channel=7, H, W)
            input_tensor[0, 0, 0:3] = norm_state
            input_tensor[0, 0, 3] = rain_norm
            input_tensor[0, 0, 4] = infil_norm
            input_tensor[0, 0, 5] = elev_norm
            input_tensor[0, 0, 6] = manning_norm

            input_torch = torch.tensor(input_tensor, dtype=torch.float32).to(device)
            pred_norm = model(input_torch).cpu().numpy()[0]  # (3, 50, 50)

            # Denormalize output state
            pred_denorm = np.zeros_like(pred_norm)
            for c in range(3):
                pred_denorm[c] = pred_norm[c] * (state_std[c] + 1e-8) + state_mean[c]

            # Post-processing: keep depth non-negative
            pred_denorm[0] = np.clip(pred_denorm[0], 0, None)

            # Save predicted state and feed recursively to next step
            pred_states.append(pred_denorm)
            current_state = pred_denorm

    pred_states = np.stack(pred_states, axis=0)  # (61, 3, 50, 50)
    return pred_states, gt_sequence, wet_mask_seq

# Evaluate on first validation scenario (index 36 is the first validation scenario since 36 are train)
pred_seq, gt_seq, wet_mask = run_rollout_evaluation(model_lstm, data, scenario_idx=36, steps=60)

# Calculate errors at 60 minutes (frame index 60)
depth_mae = np.mean(np.abs(pred_seq[60, 0] - gt_seq[60, 0]))
print(f"Validation Scenario 36 rollout (60 mins) | Depth MAE: {depth_mae:.4f} meters")

Validation Scenario 36 rollout (60 mins) | Depth MAE: 0.0774 meters


## 5. Metrics and Rollout Performance Suite
We will now write an evaluation suite that assesses predictions at 1, 5, 15, 30, and 60 minutes (corresponding to frame indices 1, 5, 15, 30, and 60). This evaluates the rollout metrics requested in Step 5.

In [18]:
def evaluate_rollout_metrics(pred_seq, gt_seq, wet_mask_seq):
    # Frame indices representing [1, 5, 15, 30, 60] minutes
    intervals = [1, 5, 15, 30, 60]
    results = {}

    for t in intervals:
        # Ground Truth and Predicted Depth
        gt_h = gt_seq[t, 0]
        pred_h = pred_seq[t, 0]

        # 1. Depth Metrics
        depth_mae = np.mean(np.abs(pred_h - gt_h))
        depth_rmse = np.sqrt(np.mean((pred_h - gt_h) ** 2))

        # 2. Critical Success Index (CSI) at 1cm threshold (0.01m)
        gt_wet = gt_h >= 0.01
        pred_wet = pred_h >= 0.01

        intersection = np.logical_and(gt_wet, pred_wet).sum()
        union = np.logical_or(gt_wet, pred_wet).sum()
        csi = intersection / union if union > 0 else 1.0

        # 3. Velocity errors where depth >= 1cm
        # To avoid division by zero, we compute velocities safely
        gt_u = np.zeros_like(gt_h)
        gt_v = np.zeros_like(gt_h)
        pred_u = np.zeros_like(pred_h)
        pred_v = np.zeros_like(pred_h)

        # Mask where depth is wet in respective states
        gt_wet_mask = gt_h >= 0.01
        pred_wet_mask = pred_h >= 0.01

        gt_u[gt_wet_mask] = gt_seq[t, 1][gt_wet_mask] / gt_h[gt_wet_mask]
        gt_v[gt_wet_mask] = gt_seq[t, 2][gt_wet_mask] / gt_h[gt_wet_mask]

        pred_u[pred_wet_mask] = pred_seq[t, 1][pred_wet_mask] / pred_h[pred_wet_mask]
        pred_v[pred_wet_mask] = pred_seq[t, 2][pred_wet_mask] / pred_h[pred_wet_mask]

        # Shared wet area to compute velocity metrics
        shared_wet = np.logical_and(gt_wet_mask, pred_wet_mask)
        if shared_wet.sum() > 0:
            vel_error = np.sqrt((pred_u[shared_wet] - gt_u[shared_wet])**2 + (pred_v[shared_wet] - gt_v[shared_wet])**2)
            vel_mae = np.mean(vel_error)
        else:
            vel_mae = 0.0

        results[t] = {
            "Depth MAE (m)": depth_mae,
            "Depth RMSE (m)": depth_rmse,
            "CSI (1cm threshold)": csi,
            "Velocity MAE (m/s)": vel_mae
        }
    return results

# Run metric suite evaluation on our scenario 36 prediction
metrics_by_time = evaluate_rollout_metrics(pred_seq, gt_seq, wet_mask)

print(f"{'-'*40}\nRollout Evaluation Metrics Over Time:\n{'-'*40}")
for t, metrics in metrics_by_time.items():
    print(f"Minute {t:2d} -> Depth MAE: {metrics['Depth MAE (m)']:.5f}m | RMSE: {metrics['Depth RMSE (m)']:.5f}m | CSI: {metrics['CSI (1cm threshold)']:.3f} | Vel MAE: {metrics['Velocity MAE (m/s)']:.4f}m/s")

----------------------------------------
Rollout Evaluation Metrics Over Time:
----------------------------------------
Minute  1 -> Depth MAE: 0.00572m | RMSE: 0.00698m | CSI: 0.000 | Vel MAE: 0.0000m/s
Minute  5 -> Depth MAE: 0.00923m | RMSE: 0.01264m | CSI: 0.000 | Vel MAE: 0.0000m/s
Minute 15 -> Depth MAE: 0.01422m | RMSE: 0.03059m | CSI: 0.305 | Vel MAE: 0.0663m/s
Minute 30 -> Depth MAE: 0.04739m | RMSE: 0.08082m | CSI: 0.376 | Vel MAE: 0.0962m/s
Minute 60 -> Depth MAE: 0.07737m | RMSE: 0.13705m | CSI: 0.372 | Vel MAE: 0.0953m/s


## 6. Analyzing ConvLSTM vs. Physics Model
Here, we calculate the absolute and relative errors of our surrogate predictions compared to the ground truth physics-simulated outputs (ANUGA). This allows us to assess how closely the neural network approximates the conservation laws and detailed flow dynamics.

In [19]:
import numpy as np

# Calculate relative error in water depth for wet cells (depth >= 1cm in Ground Truth)
gt_depth_60 = gt_seq[60, 0]
pred_depth_60 = pred_seq[60, 0]

# Mask of cells where ANUGA physics model simulated water >= 1cm
physics_wet_mask = gt_depth_60 >= 0.01

if physics_wet_mask.sum() > 0:
    abs_diff = np.abs(pred_depth_60[physics_wet_mask] - gt_depth_60[physics_wet_mask])
    mean_relative_error_pct = np.mean(abs_diff / gt_depth_60[physics_wet_mask]) * 100
    mean_abs_depth_error = np.mean(abs_diff)
else:
    mean_relative_error_pct = 0.0
    mean_abs_depth_error = 0.0

print(f"--- 60-Minute Rollout Comparison vs. ANUGA Physics Model ---")
print(f"Mean Absolute Depth Error (Wet Cells): {mean_abs_depth_error:.4f} meters")
print(f"Mean Relative Depth Error (Wet Cells): {mean_relative_error_pct:.2f}%")
print(f"Critical Success Index (CSI): {metrics_by_time[60]['CSI (1cm threshold)'] * 100:.1f}%")
print(f"Velocity Error (Wet Cells): {metrics_by_time[60]['Velocity MAE (m/s)']:.4f} m/s")

--- 60-Minute Rollout Comparison vs. ANUGA Physics Model ---
Mean Absolute Depth Error (Wet Cells): 0.1306 meters
Mean Relative Depth Error (Wet Cells): 232.43%
Critical Success Index (CSI): 37.2%
Velocity Error (Wet Cells): 0.0953 m/s


## 7. Package and Save Model Artifacts
In this step, we will create a dedicated directory called `model_artifacts` and save our model weights along with the normalization statistics so that everything is packaged in a clean, self-contained folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [20]:
import os

# Create directory
output_dir = './model_artifacts'
os.makedirs(output_dir, exist_ok=True)

# Save model weights
model_path = os.path.join(output_dir, 'convlstm_surrogate.pt')
torch.save(model_lstm.state_dict(), model_path)

# Save normalization metadata for inference pipeline compatibility
norm_path = os.path.join(output_dir, 'normalization_config.npz')
np.savez(
    norm_path,
    state_mean=state_mean,
    state_std=state_std,
    rain_mean=rain_mean,
    rain_std=rain_std,
    infil_mean=infil_mean,
    infil_std=infil_std,
    elev_mean=elev_mean,
    elev_std=elev_std,
    mann_mean=mann_mean,
    mann_std=mann_std
)

print(f"All model components successfully packaged and saved in: '{output_dir}/'")
print(os.listdir(output_dir))

All model components successfully packaged and saved in: './model_artifacts/'
['normalization_config.npz', 'convlstm_surrogate.pt']


## 8. Download Model Artifacts to Your Desktop
Run this cell to prompt your web browser to download the saved model weights and normalization config files directly to your computer.

In [21]:
from google.colab import files
import os

# Trigger files download for both packaged artifacts
if os.path.exists('/content/model_artifacts/convlstm_surrogate.pt'):
    print("Downloading model weights...")
    files.download('/content/model_artifacts/convlstm_surrogate.pt')
else:
    print("Weights file not found!")

if os.path.exists('/content/model_artifacts/normalization_config.npz'):
    print("Downloading normalization config...")
    files.download('/content/model_artifacts/normalization_config.npz')
else:
    print("Normalization config not found!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>